In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import glob
import time
import gget
import scipy
from scipy.sparse import csr_matrix
import anndata as an
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import random
from importlib import reload
import warnings
import ot
from scipy.spatial.distance import pdist, squareform
from matplotlib.colors import ListedColormap

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler

"""WARNING: no warnings"""
warnings.filterwarnings("ignore")

# local imports

source_path = os.path.abspath("../utilities/")
sys.path.append(source_path)
# import centrality as central
import matrix
import utils as ut

In [ ]:
%%time 
# 1MB resolution
resolution = 1000000

fpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/anndata/population_mESC_{resolution}_features.h5ad"

adata = sc.read_h5ad(fpath)

sc.logging.print_memory_usage()

adata

In [ ]:
def find_outliers_iqr(df_column):
  """
  Identifies outliers in a pandas DataFrame column using the IQR method.

  Args:
    df_column: A pandas Series representing the column to analyze.

  Returns:
    A boolean mask with True for outliers and False otherwise.
  """
  Q1 = df_column.quantile(0.15)
  Q3 = df_column.quantile(0.85)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  return (df_column < lower_bound) | (df_column > upper_bound)

adata.obs['degree_outlier'] = find_outliers_iqr(adata.obs['degree'])

adata.obs[adata.obs['degree_outlier']][['chrom_bin', 'degree', 'degree_outlier']].head()

In [ ]:
# remove outliers
remove_bins = adata.obs[adata.obs['degree_outlier']].index.to_list()
print(f"Removing {len(remove_bins)} outlier loci: ")
print(remove_bins)

adata = adata[~adata.obs_names.isin(remove_bins), :].copy()
#bdata = bdata[~bdata.obs_names.isin(remove_bins), :].copy()

print('done!')

In [ ]:
fpath = f"/nfs/turbo/umms-indikar/shared/projects/poreC/pipeline_outputs/higher_order/global_core_score/pop_mESC_1000000_hge_logexp.csv"

df = pd.read_csv(fpath)
print(f"{df.shape=}")
print(df.columns)

In [ ]:
#columns_to_drop = [x for x in df.columns if x in adata.obs.columns]
columns_to_drop = ["chrom", "bin_start", "bin_end"]
df = df.drop(columns=columns_to_drop)
print(f"{df.shape=}")
df.head()

In [ ]:
adata.obs = pd.merge(
    adata.obs, df.set_index('bin_name'),
    how='left',
    left_index=True,
    right_index=True,
)

adata.obs.head()

In [ ]:
pdf = adata.obs.copy()

column_labels = {
    'n_genes': "genes",
    'RNA_5': "RNA",
    'ATACSeq_3': "ATAC-seq",
    'CTCF': "CTCF",
    'H3K27ac': "H3K27ac",
    'H3K27me3': "H3K27me3",
    #'ce_degree_centrality': "degree",
    #'ce_eigenvector_centrality': "eigenvector",
    #'ce_betweenness_centrality': "betweenness",
    # 'singular_vector_1': "singular vector",
    #'hge_linear_unweighted' : "linear",
    'global_hge_logexp_unweighted' : "log-exp",
    #'hge_logexp_RNA_weighted' : "log-exp (RNA)",
    #'hge_logexp_ATAC_weighted' : "log-exp (ATAC)",
}

corr = pdf[list(column_labels.keys())].corr()
corr.index = list(column_labels.values())
corr.columns = list(column_labels.values())

plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = 7, 7

sns.heatmap(
    corr, 
    square=True,
    linewidths=1,
    cmap='coolwarm',
    annot=True,
    fmt=".2f",  
    annot_kws={"size": 7.5},
    cbar_kws={'shrink': 0.3, 'label' : 'correlation'}
)

# Get the positions of the lines
# These positions are based on the order of your `column_labels`
lines = [6, 9,]

# Add vertical lines
for line in lines:
    plt.axvline(line, color='black', lw=2)
    plt.axhline(line, color='black', lw=2)

plt.show()